## 1. Configuration and Imports

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    col,
    count,
    countDistinct,
    current_timestamp,
    desc,
    input_file_name,
    max,
    min,
    to_timestamp,
    when,
)
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
import os

## 2. Create Spark Session with Delta Lake

In [15]:
# Path configuration
jar_path = os.path.abspath("jars/delta-spark_2.12-3.2.1.jar") + "," + os.path.abspath("jars/delta-storage-3.2.1.jar")

spark = SparkSession.builder \
    .appName("Bronze_Pipeline_JSON_to_Delta") \
    .config("spark.jars", jar_path) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.streaming.schemaInference", "true") \
    .getOrCreate()  # type: ignore[attr-defined]

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Session created (version {spark.version})")

Spark Session created (version 3.5.3)


## 3. Path Configuration

In [16]:
# Paths for Medallion architecture
INPUT_PATH = "data/sensor_data"
OUTPUT_PATH = "output/delta/bronze/sensor_data"
CHECKPOINT_PATH = "checkpoints/bronze"

print(f"Input Path: {INPUT_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print(f"Checkpoint Path: {CHECKPOINT_PATH}")

Input Path: data/sensor_data
Output Path: output/delta/bronze/sensor_data
Checkpoint Path: checkpoints/bronze


## 4. IoT Sensor Data Schema

Sensor measurement structure:
- `timestamp`: ISO 8601 timestamp
- `device_id`: Unique sensor identifier
- `building`: Building (A, B, C)
- `floor`: Floor number
- `type`: Measurement type (temperature, humidity, co2)
- `value`: Measured value
- `unit`: Unit (°C, %, ppm)

In [17]:
sensor_schema = StructType([
    StructField("timestamp", StringType(), False),
    StructField("device_id", StringType(), False),
    StructField("building", StringType(), False),
    StructField("floor", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("value", DoubleType(), False),
    StructField("unit", StringType(), False)
])

print("Schema defined")

Schema defined


## 5. Read JSON Stream

Using `readStream` to process JSON files continuously.

In [18]:
json_stream = spark.readStream \
    .schema(sensor_schema) \
    .option("maxFilesPerTrigger", 5) \
    .json(INPUT_PATH)

print("JSON stream configured")
print(f"Schema: {json_stream.schema}")

JSON stream configured
Schema: StructType([StructField('timestamp', StringType(), True), StructField('device_id', StringType(), True), StructField('building', StringType(), True), StructField('floor', IntegerType(), True), StructField('type', StringType(), True), StructField('value', DoubleType(), True), StructField('unit', StringType(), True)])


## 6. Bronze Transformations

Raw data enrichment:
1. **Timestamp conversion**: String → Timestamp
2. **Ingestion timestamp**: Processing time
3. **Source file**: Source file traceability
4. **Anomaly detection**:
   - CO₂ > 1000 ppm
   - Temperature < 15°C or > 30°C
   - Humidity < 20% or > 80%
5. **Data quality**: Completeness of required fields

In [19]:
bronze_stream = json_stream \
    .withColumn("event_timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name()) \
    .withColumn(
        "anomaly_detected",
        when((col("type") == "co2") & (col("value") > 1000), True)
        .when((col("type") == "temperature") & ((col("value") < 15) | (col("value") > 30)), True)
        .when((col("type") == "humidity") & ((col("value") < 20) | (col("value") > 80)), True)
        .otherwise(False)
    ) \
    .withColumn(
        "data_quality",
        when(
            col("device_id").isNotNull() & 
            col("building").isNotNull() & 
            col("value").isNotNull(),
            "complete"
        ).otherwise("incomplete")
    ) \
    .select(
        col("device_id"),
        col("building"),
        col("floor"),
        col("type"),
        col("value"),
        col("unit"),
        col("event_timestamp"),
        col("ingestion_timestamp"),
        col("anomaly_detected"),
        col("data_quality"),
        col("source_file")
    )

print("Bronze transformations configured")

Bronze transformations configured


## 7. Write to Delta Lake

Streaming configuration to Delta Lake:
- **Format**: Delta Lake (ACID + Versioning)
- **Mode**: Append (continuous append)
- **Checkpoint**: Fault tolerance
- **Trigger**: Process every 10 seconds

In [20]:
query = bronze_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="10 seconds") \
    .start(OUTPUT_PATH)

print("Streaming query started")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")

Streaming query started
Query ID: ddccd4a4-3f94-4331-9e5d-8794894acc63
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


25/12/17 15:15:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 8. Streaming Monitoring

Monitoring processing metrics.

In [21]:
import time

# Wait a few seconds to see processing
time.sleep(20)

print("Streaming status:")
print(f"Is Active: {query.isActive}")
print(f"Recent Progress: {len(query.recentProgress)} batches")

if query.recentProgress:
    latest = query.recentProgress[-1]
    print("\nLatest batch:")
    print(f"  - Batch ID: {latest.get('batchId', 'N/A')}")
    print(f"  - Input Rows: {latest.get('numInputRows', 0)}")
    print(f"  - Process Rate: {latest.get('processedRowsPerSecond', 0):.2f} rows/sec")

25/12/17 15:15:15 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_99.json was not found. Was it deleted very recently?
25/12/17 15:15:15 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_76.json was not found. Was it deleted very recently?
25/12/17 15:15:15 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_60.json was not found. Was it deleted very recently?
25/12/17 15:15:15 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_37.json was not found. Was it deleted very recently?
25/12/17 15:15:15 WARN HadoopFSUtils: The directory file:/app/data/sensor_data/sensor_data_102.json was not found. Was it deleted very recently?
25/12/17 15:15:15 ERROR MicroBatchExecution: Query [id = ddccd4a4-3f94-4331-9e5d-8794894acc63, runId = 689d4eb0-6f6e-452c-9649-f68138bd17b5] terminated with error
java.lang.IllegalArgumentException: Wrong basePath data/sensor_data for the root path: file:/app/data/sensor_data/se

Streaming status:
Is Active: False
Recent Progress: 0 batches


## 9. Bronze Data Verification

Batch read for written data validation.

In [22]:
# Read Bronze data
bronze_df = spark.read.format("delta").load(OUTPUT_PATH)

print(f"Total Bronze records: {bronze_df.count()}")
print("\nData preview:")
bronze_df.show(10, truncate=False)

Total Bronze records: 500

Data preview:
+-----------------+--------+-----+------------------+-----+----+-------------------+-----------------------+----------------+------------+------------------------------------------------+
|device_id        |building|floor|type              |value|unit|event_timestamp    |ingestion_timestamp    |anomaly_detected|data_quality|source_file                                     |
+-----------------+--------+-----+------------------+-----+----+-------------------+-----------------------+----------------+------------+------------------------------------------------+
|sensor-hum-002   |B       |2    |humidity          |51.3 |%   |2025-01-12 09:30:27|2025-12-17 14:36:00.015|false           |complete    |file:///app/data/sensor_data/sensor_data_55.json|
|sensor-energy-010|A       |1    |energy_consumption|112.9|kWh |2025-01-12 09:30:32|2025-12-17 14:36:00.015|false           |complete    |file:///app/data/sensor_data/sensor_data_55.json|
|sensor-temp-003  |

In [23]:
# Statistics by building
print("Statistics by building:")
bronze_df.groupBy("building").agg(
    count("*").alias("total_records"),  # type: ignore[attr-defined]
    countDistinct("device_id").alias("unique_devices")
).show()

Statistics by building:


+--------+-------------+--------------+
|building|total_records|unique_devices|
+--------+-------------+--------------+
|       B|          256|             5|
|       A|          244|             5|
+--------+-------------+--------------+



In [24]:
# Statistics by sensor type
print("Statistics by sensor type:")
bronze_df.groupBy("type").agg(
    count("*").alias("total_records"),  # type: ignore[attr-defined]
    avg("value").alias("avg_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value")
).show()

Statistics by sensor type:
+------------------+-------------+------------------+---------+---------+
|              type|total_records|         avg_value|min_value|max_value|
+------------------+-------------+------------------+---------+---------+
|          humidity|           94| 44.78191489361702|     30.1|     59.6|
|       temperature|          215| 23.46511627906977|     18.1|     27.9|
|energy_consumption|           79|150.07848101265822|    102.4|    198.9|
|               co2|          112| 809.0267857142857|    400.0|   1200.0|
+------------------+-------------+------------------+---------+---------+

+------------------+-------------+------------------+---------+---------+
|              type|total_records|         avg_value|min_value|max_value|
+------------------+-------------+------------------+---------+---------+
|          humidity|           94| 44.78191489361702|     30.1|     59.6|
|       temperature|          215| 23.46511627906977|     18.1|     27.9|
|energy_co

In [25]:
# Detected anomalies
print(" Detected anomalies:")
anomalies = bronze_df.filter(col("anomaly_detected") == True)
print(f"Total anomalies: {anomalies.count()}")
anomalies.groupBy("type", "building").count().orderBy(desc("count")).show()

 Detected anomalies:
Total anomalies: 33
Total anomalies: 33
+----+--------+-----+
|type|building|count|
+----+--------+-----+
| co2|       B|   19|
| co2|       A|   14|
+----+--------+-----+

+----+--------+-----+
|type|building|count|
+----+--------+-----+
| co2|       B|   19|
| co2|       A|   14|
+----+--------+-----+



## 10. Delta Lake History

Versioning and transaction verification.

In [26]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, OUTPUT_PATH)
print("Transaction history:")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(10, truncate=False)

Transaction history:
+-------+-----------------------+----------------+---------------------------------------------------------------------------------------+
|version|timestamp              |operation       |operationMetrics                                                                       |
+-------+-----------------------+----------------+---------------------------------------------------------------------------------------+
|20     |2025-12-17 14:38:50.923|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 7540, numAddedFiles -> 2}|
|19     |2025-12-17 14:38:41.06 |STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 7519, numAddedFiles -> 2}|
|18     |2025-12-17 14:38:30.901|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 7523, numAddedFiles -> 2}|
|17     |2025-12-17 14:38:20.886|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 7527, numAddedFiles -> 2}|
|16   

## 11. Stop Streaming

In [27]:
# Stop streaming query
query.stop()
time.sleep(2)
print(f"Streaming stopped (Active: {query.isActive})")

Streaming stopped (Active: False)


## Key Concepts Illustrated

### 1. Spark Structured Streaming
- Stream processing as infinite tables
- Unified API for batch and streaming

### 2. Checkpointing
- Streaming state persistence
- Exact recovery after failure
- Fault tolerance guarantee

### 3. Append Mode
- Only new rows are written
- Ideal for immutable data (Bronze)

### 4. Triggers
- `processingTime`: Regular intervals (10s)
- Others: `once`, `continuous`

### 5. Delta Lake (Bronze)
- ACID transactions
- Versioning and time travel
- Raw data with metadata

### 6. Medallion Architecture
- **Bronze**: Raw storage with minimal validation
- Source of truth for Silver/Gold